In [1]:
from pathlib import Path
import json
import math
from datetime import date, timedelta
from collections import defaultdict
from typing import Dict, List, Tuple, Any, Optional
import zipfile

import pandas as pd
import numpy as np

# =========================
# 0. PATHS & CONFIG
# =========================

PROJECT = Path.cwd()

RAW = PROJECT / "data" / "raw" / "tomtom_stats"
GRAPH_DIR = RAW / "graphs"
TIME_DIR = RAW / "times"
GRAPH_DIR.mkdir(parents=True, exist_ok=True)
TIME_DIR.mkdir(parents=True, exist_ok=True)

# --- NEW: ZIP input + extracted folder ---
INPUT_ZIP = RAW / "tomtom_traffic.zip"         # file zip bạn có
EXTRACT_DIR = RAW / "tomtom_traffic"           # folder giải nén (đúng yêu cầu)

# output CSV giống các notebook cũ
NODE_CSV          = GRAPH_DIR / "node.csv"
SEGMENTS_CSV      = GRAPH_DIR / "segments.csv"
EDGES_CSV         = GRAPH_DIR / "edges.csv"
SEGMENT_INDEX_CSV = GRAPH_DIR / "segment_index.csv"
OBSERVATIONS_CSV  = TIME_DIR / "observations.csv"

# Lọc theo FRC nếu muốn, vd: {6,7} hoặc None để giữ tất cả
TARGET_FRC: Optional[set] = None

# tham số spatial
NODE_SNAP_METERS = 5.0   # giống THERSHOLD_METERS trong spatial notebook
GRID_METERS      = 10.0


# =========================
# 1. HÀM HỖ TRỢ
# =========================

def haversine_m(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    """Khoảng cách theo đường tròn lớn (m) giữa 2 điểm (lat, lon)."""
    R = 6371000.0
    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)
    dphi = phi2 - phi1
    dlambda = math.radians(lon2 - lon1)

    a = math.sin(dphi / 2) ** 2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2) ** 2
    return 2 * R * math.asin(math.sqrt(a))


def cell_key(lat: float, lon: float, meters: float = GRID_METERS) -> Tuple[int, int]:
    """Gán (lat, lon) vào ô lưới ~meters mét – giống notebook spatial."""
    dlat = meters / 111320.0
    dlon = meters / (111320.0 * max(0.2, math.cos(math.radians(lat))))
    return int(lat / dlat), int(lon / dlon)


def _percentiles(sp: List[int]) -> Tuple[Any, Any, Any, Any, Any]:
    """
    Lấy p05, p25, p50, p75, p95 từ list 19 phần trăm tốc độ của TomTom,
    giữ đúng mapping như flatten_timeseries cũ.
    """
    return (
        sp[2]  if len(sp) >= 3  else "",
        sp[5]  if len(sp) >= 6  else "",
        sp[9]  if len(sp) >= 10 else "",
        sp[13] if len(sp) >= 14 else "",
        sp[17] if len(sp) >= 18 else "",
    )


def _list_days(dr: dict) -> List[Tuple[str, str]]:
    """
    Từ 1 dateRange của TomTom → list (date_iso, DAY_OF_WEEK),
    dùng 'from'–'to' và loại các 'excludedDaysOfWeek'.
    """
    start = dr.get("from")
    end   = dr.get("to")
    if not start or not end:
        return []

    y1, m1, d1 = map(int, start.split("-"))
    y2, m2, d2 = map(int, end.split("-"))

    cur, stop = date(y1, m1, d1), date(y2, m2, d2)
    excluded = set(dr.get("excludedDaysOfWeek", []))

    out = []
    while cur <= stop:
        dow = cur.strftime("%A").upper()
        if dow not in excluded:
            out.append((cur.isoformat(), dow))
        cur += timedelta(days=1)
    return out


# =========================
# 2. ZIP: EXTRACT & LIST JSON FILES
# =========================

def ensure_extracted(zip_path: Path, extract_dir: Path) -> None:
    """
    Giải nén zip vào extract_dir nếu:
    - extract_dir chưa tồn tại, hoặc
    - extract_dir không có file .json nào
    """
    extract_dir.mkdir(parents=True, exist_ok=True)

    existing = list(extract_dir.rglob("*.json"))
    if existing:
        return

    if not zip_path.exists():
        raise FileNotFoundError(f"Không thấy file zip: {zip_path}")

    print(f">>> Giải nén: {zip_path} -> {extract_dir}")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(extract_dir)


def list_json_files(extract_dir: Path) -> List[Path]:
    """
    Lấy danh sách các file json trong folder đã giải nén.
    Hỗ trợ trường hợp zip chứa folder con (rglob).
    """
    files = sorted(extract_dir.rglob("*.json"))
    if not files:
        raise FileNotFoundError(f"Không tìm thấy .json nào trong: {extract_dir}")
    return files


# =========================
# 3. ĐỌC JSON TOMTOM STATS (giữ nguyên)
# =========================

def load_tomtom_stats(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    if "network" not in data or "segmentResults" not in data["network"]:
        raise ValueError(f"JSON không có network.segmentResults – check lại file: {path}")

    segment_results = data["network"]["segmentResults"]
    time_sets_raw  = data.get("timeSets", [])
    date_ranges_raw = data.get("dateRanges", [])
    zone_id = data["network"].get("zoneId")

    time_sets: Dict[int, dict] = {}
    for ts in time_sets_raw:
        tid = ts.get("@id")
        if tid is not None:
            time_sets[tid] = ts

    date_ranges: Dict[int, dict] = {}
    for dr in date_ranges_raw:
        drid = dr.get("@id")
        if drid is not None:
            date_ranges[drid] = dr

    return segment_results, time_sets, date_ranges, zone_id


# =========================
# 4. BUILD GRAPH (giữ nguyên)
# =========================

def build_spatial_graph(segment_results: List[dict]):
    # 4.1. Lấy endpoints từng segment
    seg_endpoints: Dict[int, Tuple[Tuple[float, float], Tuple[float, float]]] = {}
    props: Dict[int, dict] = {}

    for seg in segment_results:
        sid = seg["segmentId"]
        if TARGET_FRC is not None and seg.get("frc") not in TARGET_FRC:
            continue

        shp = seg.get("shape") or []
        if len(shp) < 2:
            continue

        a = shp[0]
        b = shp[-1]
        seg_endpoints[sid] = ((a["latitude"], a["longitude"]), (b["latitude"], b["longitude"]))
        props[sid] = seg

    # 4.2. Gom endpoint -> node
    grid = defaultdict(list)   # cell_key -> list (node_id,lat,lon)
    nodes: Dict[int, Tuple[float, float]] = {}
    node_of_endpoint: Dict[Tuple[int, int], int] = {}
    next_node_id = 0

    for sid, ends in seg_endpoints.items():
        for ei, (lat, lon) in enumerate(ends):
            ck = cell_key(lat, lon, meters=GRID_METERS)

            cand = []
            for di in [-1, 0, 1]:
                for dj in [-1, 0, 1]:
                    cand.extend(grid.get((ck[0] + di, ck[1] + dj), []))

            assigned = None
            for nid, nlat, nlon in cand:
                if haversine_m(lat, lon, nlat, nlon) <= NODE_SNAP_METERS:
                    assigned = nid
                    break

            if assigned is None:
                nid = next_node_id
                next_node_id += 1
                nodes[nid] = (lat, lon)
                grid[ck].append((nid, lat, lon))
                assigned = nid

            node_of_endpoint[(sid, ei)] = assigned

    # 4.3. seg_to_nodes
    seg_to_nodes: Dict[int, Tuple[int, int]] = {}
    for sid in seg_endpoints.keys():
        u = node_of_endpoint[(sid, 0)]
        v = node_of_endpoint[(sid, 1)]
        if u != v:
            seg_to_nodes[sid] = (u, v)

    # 4.4. node_to_segs
    node_to_segs: Dict[int, set] = defaultdict(set)
    for sid, (u, v) in seg_to_nodes.items():
        node_to_segs[u].add(sid)
        node_to_segs[v].add(sid)

    # 4.5. seg_edges
    seg_edges: set = set()
    for segs in node_to_segs.values():
        segs = sorted(segs)
        for i in range(len(segs)):
            for j in range(i + 1, len(segs)):
                a, b = segs[i], segs[j]
                if a != b:
                    seg_edges.add((a, b))

    # 4.6. node.csv
    node_rows = []
    for nid, (lat, lon) in nodes.items():
        node_rows.append({
            "node_id": nid,
            "lat": f"{lat:.7f}",
            "lon": f"{lon:.7f}",
            "segments_count": len(node_to_segs.get(nid, []))
        })
    node_df = pd.DataFrame(node_rows)

    # 4.7. segments.csv
    seg_rows = []
    for sid, (u, v) in seg_to_nodes.items():
        p = props[sid]
        seg_rows.append({
            "segment_id": sid,
            "node_u": u,
            "node_v": v,
            "streetName": p.get("streetName", ""),
            "frc": p.get("frc", ""),
            "speedLimit": p.get("speedLimit", ""),
            "distance": p.get("distance", ""),
        })
    segments_df = pd.DataFrame(seg_rows)

    # 4.8. edges.csv
    edge_rows = [{"segment_u": a, "segment_v": b} for (a, b) in sorted(seg_edges)]
    edges_df = pd.DataFrame(edge_rows)

    # 4.9. segment_index.csv
    seg_ids = sorted(seg_to_nodes.keys())
    index = {sid: i for i, sid in enumerate(seg_ids)}
    seg_index_rows = [{"idx": idx, "segment_id": sid} for sid, idx in index.items()]
    seg_index_df = pd.DataFrame(seg_index_rows)

    return node_df, segments_df, edges_df, seg_index_df


# =========================
# 5. BUILD observations.csv (giữ nguyên)
# =========================

def build_observations(
    segment_results: List[dict],
    time_sets: Dict[int, dict],
    date_ranges: Dict[int, dict],
):
    days_by_dr: Dict[int, List[Tuple[str, str]]] = {
        drid: _list_days(dr) for drid, dr in date_ranges.items()
    }

    timeset_to_valid_dow: Dict[int, set] = {}
    timeset_to_timerange: Dict[int, str] = {}

    for tid, ts in time_sets.items():
        dows = {d["dayOfWeek"].upper() for d in ts.get("dayToTimeRanges", [])}
        tr = None
        for d in ts.get("dayToTimeRanges", []):
            trs = d.get("timeRanges", [])
            if trs:
                tr = trs[0]
                break
        if tr is None:
            continue
        timeset_to_valid_dow[tid] = dows
        timeset_to_timerange[tid] = tr

    rows = []

    for seg in segment_results:
        sid = seg["segmentId"]
        new_sid = seg.get("newSegmentId")
        frc = seg.get("frc")
        if TARGET_FRC is not None and frc not in TARGET_FRC:
            continue

        street = seg.get("streetName")
        speed_limit = seg.get("speedLimit")
        dist = seg.get("distance")

        for tr in seg.get("segmentTimeResults", []):
            tid = tr.get("timeSet")
            drid = tr.get("dateRange")
            if tid is None or drid is None:
                continue
            trange = timeset_to_timerange.get(tid)
            if not trange or "-" not in trange:
                continue
            time_start, time_end = trange.split("-")
            valid_dow = timeset_to_valid_dow.get(tid, set())

            sp = tr.get("speedPercentiles") or []
            p05, p25, p50, p75, p95 = _percentiles(sp)

            sample = tr.get("sampleSize", 0) or 0
            is_obs = 1 if sample > 0 else 0

            days = days_by_dr.get(drid, [])
            if not days:
                rows.append([
                    "", "", tid, time_start, time_end, sid, new_sid,
                    street, frc, speed_limit, dist, sample,
                    tr.get("averageSpeed", ""),
                    tr.get("medianSpeed", ""),
                    tr.get("harmonicAverageSpeed", ""),
                    tr.get("averageTravelTime", ""),
                    tr.get("medianTravelTime", ""),
                    tr.get("travelTimeRatio", ""),
                    p05, p25, p50, p75, p95, is_obs
                ])
                continue

            for dt, dow in days:
                if dow not in valid_dow:
                    continue
                rows.append([
                    dt, dow, tid, time_start, time_end, sid, new_sid,
                    street, frc, speed_limit, dist, sample,
                    tr.get("averageSpeed", ""),
                    tr.get("medianSpeed", ""),
                    tr.get("harmonicAverageSpeed", ""),
                    tr.get("averageTravelTime", ""),
                    tr.get("medianTravelTime", ""),
                    tr.get("travelTimeRatio", ""),
                    p05, p25, p50, p75, p95, is_obs
                ])

    obs_df = pd.DataFrame(
        rows,
        columns=[
            "date", "day_of_week",
            "time_set_id", "time_start", "time_end",
            "segment_id", "newSegmentId",
            "streetName", "frc", "speedLimit", "distance",
            "sampleSize",
            "averageSpeed", "medianSpeed", "harmonicAverageSpeed",
            "averageTravelTime", "medianTravelTime", "travelTimeRatio",
            "p05", "p25", "p50", "p75", "p95",
            "is_observed",
        ]
    )
    return obs_df


# =========================
# 6. MAIN (NEW: loop all json days)
# =========================

def main():
    # 6.1 Extract zip -> folder tomtom_traffic/
    ensure_extracted(INPUT_ZIP, EXTRACT_DIR)
    json_files = list_json_files(EXTRACT_DIR)

    print(f">>> Tìm thấy {len(json_files)} file JSON trong {EXTRACT_DIR}")

    # 6.2 Build graph ONCE from first JSON (giữ nguyên ý nghĩa)
    first_json = json_files[0]
    print(">>> Build graph từ file đầu tiên:", first_json.name)
    seg_results0, time_sets0, date_ranges0, zone_id0 = load_tomtom_stats(first_json)
    print(f"[INFO] Zone ID: {zone_id0}")
    print(f"[INFO] Segments (first day): {len(seg_results0)}")

    node_df, segments_df, edges_df, seg_index_df = build_spatial_graph(seg_results0)

    node_df.to_csv(NODE_CSV, index=False, encoding="utf-8")
    segments_df.to_csv(SEGMENTS_CSV, index=False, encoding="utf-8")
    edges_df.to_csv(EDGES_CSV, index=False, encoding="utf-8")
    seg_index_df.to_csv(SEGMENT_INDEX_CSV, index=False, encoding="utf-8")

    print(f"[OK] Saved node.csv           → {NODE_CSV}")
    print(f"[OK] Saved segments.csv       → {SEGMENTS_CSV}")
    print(f"[OK] Saved edges.csv          → {EDGES_CSV}")
    print(f"[OK] Saved segment_index.csv  → {SEGMENT_INDEX_CSV}")

    # 6.3 Build observations by concatenating all days
    print(">>> Flatten observations cho từng ngày và concat ...")
    obs_all = []

    # (optional) dùng set segmentId để cảnh báo nếu ngày nào đó lệch network
    base_seg_ids = {s["segmentId"] for s in seg_results0}

    for i, jf in enumerate(json_files, start=1):
        seg_results, time_sets, date_ranges, zone_id = load_tomtom_stats(jf)

        cur_seg_ids = {s["segmentId"] for s in seg_results}
        if cur_seg_ids != base_seg_ids:
            # chỉ cảnh báo, vẫn chạy tiếp để bạn không bị “đứt pipeline”
            print(f"[WARN] Segment set khác với ngày đầu tiên ở file: {jf.name} "
                  f"(base={len(base_seg_ids)} vs cur={len(cur_seg_ids)})")

        obs_df = build_observations(seg_results, time_sets, date_ranges)

        # (optional) nếu muốn truy vết file nguồn, bật 2 dòng dưới
        # obs_df["source_file"] = jf.name
        # obs_df["source_idx"] = i

        obs_all.append(obs_df)
        print(f"  - ({i}/{len(json_files)}) {jf.name}: rows={len(obs_df)}")

    obs_all_df = pd.concat(obs_all, ignore_index=True) if obs_all else pd.DataFrame()

    obs_all_df.to_csv(OBSERVATIONS_CSV, index=False, encoding="utf-8")
    print(f"[OK] Saved observations.csv   → {OBSERVATIONS_CSV}")
    print("[DONE] Tổng số dòng observation:", len(obs_all_df))


if __name__ == "__main__":
    main()


FileNotFoundError: Không thấy file zip: C:\AI\Specialized_Project2_github\Urban-Traffic-Links\src\data_processing\tomtom\data\raw\tomtom_stats\tomtom_traffic.zip